# Subtopic 3 — Monotonic Stack / Queue

**Problems:** Next Greater Element I, Next Greater Element II (circular), Next Smaller Element, Count of Greater Elements to the Right, Trapping Rainwater, Sum of Subarray Minimums, Asteroid Collision, Sum of Subarray Ranges, Remove K Digits, Largest Rectangle in a Histogram, Maximal Rectangle.

A monotonic stack is one idea applied eleven ways: **maintain a stack whose values are sorted, and let the act of restoring that order answer a question about each element.** When an element is popped, the element that displaced it (or the one beneath it) is precisely its "next/previous greater/smaller" — the answer falls out of the pop.

Master the core invariant and the amortized bound once; every problem below is a choice of (a) increasing vs decreasing order, (b) what a pop *means*, and (c) what to do with elements still on the stack at the end.

## The Monotonic Stack Invariant

We store **indices** (not values) so we can recover both position and value, and compute distances.

> **Invariant (decreasing stack).** At all times, the values at the stacked indices are strictly decreasing from bottom to top:
> $$a[\,\text{st}[0]\,] > a[\,\text{st}[1]\,] > \cdots > a[\,\text{st}[\text{top}]\,].$$
> (Increasing stack: replace $>$ with $<$. "Non-strict" variants use $\ge$ / $\le$ — the strict/non-strict choice is how we break ties to avoid double-counting; see Sum of Subarray Minimums.)

### Maintenance
Before pushing index $i$, pop every top whose value violates the target order against $a[i]$. After the pops, pushing $i$ restores the invariant. Each pop *resolves* the popped element: the element that caused the pop is its **next greater/smaller** to the right; the new top (after popping) is its **previous greater/smaller** to the left.

### Amortized $O(n)$
Each index is **pushed exactly once** and **popped at most once**. Across the whole scan the total number of push and pop operations is therefore $\le 2n$:

$$\sum_{i} (\text{pushes}_i + \text{pops}_i) = n + (\le n) \le 2n = O(n).$$

The inner `while` loop can run many iterations on a single step, but those iterations are paid for by earlier pushes — a textbook amortized (aggregate) argument. Per-operation cost is $O(1)$ amortized; total is $O(n)$.

### What a pop tells you (the universal table)
| You scan… | Stack order | When you pop index $j$ because of $i$ | Meaning |
|---|---|---|---|
| L→R | decreasing | $a[i] \ge a[j]$ pops $j$ | $i$ = Next Greater-or-Equal of $j$ (right) |
| L→R | increasing | $a[i] \le a[j]$ pops $j$ | $i$ = Next Smaller-or-Equal of $j$ (right) |
| after popping, new top $t$ | decreasing | — | $t$ = Previous Greater of $i$ (left) |
| after popping, new top $t$ | increasing | — | $t$ = Previous Smaller of $i$ (left) |

In [ ]:
#include <iostream>
#include <vector>
#include <stack>
#include <deque>
#include <string>
#include <algorithm>
#include <climits>
#include <unordered_map>
using namespace std;

template <typename T>
void printVec(const string& label, const vector<T>& v) {
    cout << label << " [";
    for (size_t i = 0; i < v.size(); ++i) cout << v[i] << (i+1<v.size() ? "," : "");
    cout << "]";
}

## 1. Next Greater Element I

For each element, find the first strictly greater element to its right (or $-1$). NGE I additionally maps results back through a query array via a hash map, but the core is a single decreasing monotonic stack.

### Invariant
Scanning **right → left**, the stack holds (values of) candidates that are still *possible* next-greater elements for indices not yet processed — kept **decreasing from bottom to top**. Any candidate $\le$ the current element can never be the answer for anything further left, so we pop it.

### Why it works
For index $i$, after popping all stack elements $\le a[i]$, the top (if any) is the nearest element to the right that is strictly greater — exactly the next greater element. We then push $a[i]$ as a candidate for indices further left.

### Boundary transitions (R→L scan)
| At index $i$ | Action |
|---|---|
| stack empty after popping $\le a[i]$ | answer$[i] = -1$ |
| stack non-empty after popping | answer$[i] = $ stack top |
| always | push $a[i]$ |

In [ ]:
// Next Greater Element for every index (strictly greater, to the right). -1 if none.
vector<int> nextGreater(const vector<int>& a) {
    int n = a.size();
    vector<int> ans(n, -1);
    stack<int> st;                          // values, decreasing bottom->top
    for (int i = n - 1; i >= 0; --i) {      // scan right to left
        while (!st.empty() && st.top() <= a[i]) st.pop();  // drop non-greater candidates
        if (!st.empty()) ans[i] = st.top(); // nearest strictly greater on the right
        st.push(a[i]);                       // a[i] is a candidate for indices to its left
    }
    return ans;
}

// NGE I: answer queries (nums1 subset of nums2) via a value->NGE map.
vector<int> nextGreaterElementI(const vector<int>& nums1, const vector<int>& nums2) {
    vector<int> nge = nextGreater(nums2);
    unordered_map<int,int> mp;               // value -> its next greater (distinct values assumed)
    for (int i = 0; i < (int)nums2.size(); ++i) mp[nums2[i]] = nge[i];
    vector<int> res;
    for (int x : nums1) res.push_back(mp[x]);
    return res;
}

In [ ]:
// Tests: input -> actual (Expected: X)
{
    auto r1 = nextGreater({2,1,2,4,3});
    printVec("nextGreater [2,1,2,4,3] ->", r1); cout << " (Expected: [4,2,4,-1,-1])\n";

    auto r2 = nextGreaterElementI({4,1,2}, {1,3,4,2});
    printVec("NGE I {4,1,2} in {1,3,4,2} ->", r2); cout << " (Expected: [-1,3,-1])\n";

    auto r3 = nextGreater({5,4,3,2,1});      // strictly decreasing: none greater
    printVec("nextGreater [5,4,3,2,1] ->", r3); cout << " (Expected: [-1,-1,-1,-1,-1])\n";

    auto r4 = nextGreater({1,2,3,4,5});      // strictly increasing
    printVec("nextGreater [1,2,3,4,5] ->", r4); cout << " (Expected: [2,3,4,5,-1])\n";

    auto r5 = nextGreater({7});              // single element
    printVec("nextGreater [7] ->", r5); cout << " (Expected: [-1])\n";

    auto r6 = nextGreater({3,3,3});          // all-same: not STRICTLY greater
    printVec("nextGreater [3,3,3] ->", r6); cout << " (Expected: [-1,-1,-1])\n";
}

## 2. Next Greater Element II (Circular)

The array is circular: after the last element, wrapping continues to index $0$. So the next greater of the last element may be at the front.

### The array-doubling trick
Iterate the index $i$ from $2n-1$ down to $0$, but **address the array as `a[i % n]`**. This simulates two concatenated copies of the array.

### Why $2n$ iterations suffice (and no element is answered twice)
Any element's circular "next greater" lies within the next $n-1$ positions (one full lap). Scanning a window of length $2n$ guarantees every original index $i \in [0,n)$ has seen all $n-1$ elements that could follow it circularly. We **write the answer only for $i < n$** during the second half (the original indices); the first $n$ iterations ($i \in [n, 2n)$) merely seed the stack with candidates from the wrap-around tail. Each original index's answer is written exactly once. $\square$

### Boundary transitions
Identical to NGE, but loop bound is $2n$ and indexing is `i % n`; only assign `ans[i % n]` when `i < n`.

In [ ]:
vector<int> nextGreaterCircular(const vector<int>& a) {
    int n = a.size();
    vector<int> ans(n, -1);
    stack<int> st;                              // stores VALUES, decreasing
    for (int i = 2*n - 1; i >= 0; --i) {        // two laps, right to left
        int v = a[i % n];                       // wrap addressing
        while (!st.empty() && st.top() <= v) st.pop();
        if (i < n && !st.empty()) ans[i] = st.top(); // write only for original indices
        st.push(v);
    }
    return ans;
}

In [ ]:
// Tests: input -> actual (Expected: X)
{
    auto r1 = nextGreaterCircular({1,2,1});
    printVec("NGE II [1,2,1] ->", r1); cout << " (Expected: [2,-1,2])\n";  // last 1 wraps to 2
    auto r2 = nextGreaterCircular({5,4,3,2,1});
    printVec("NGE II [5,4,3,2,1] ->", r2); cout << " (Expected: [-1,5,5,5,5])\n";
    auto r3 = nextGreaterCircular({3,3,3});
    printVec("NGE II [3,3,3] ->", r3); cout << " (Expected: [-1,-1,-1])\n"; // strictly greater
    auto r4 = nextGreaterCircular({1});
    printVec("NGE II [1] ->", r4); cout << " (Expected: [-1])\n";
}

## 3. Next Smaller Element

Dual of NGE: first strictly smaller element to the right. Flip the stack order to **increasing** and the pop comparison to `>=`.

### Invariant
Scanning R→L, the stack holds candidate values **increasing bottom→top**. Pop everything $\ge a[i]$ (cannot be a *smaller* answer for anything further left); the surviving top is the next smaller.

In [ ]:
vector<int> nextSmaller(const vector<int>& a) {
    int n = a.size();
    vector<int> ans(n, -1);
    stack<int> st;                            // increasing bottom->top
    for (int i = n - 1; i >= 0; --i) {
        while (!st.empty() && st.top() >= a[i]) st.pop();  // drop non-smaller candidates
        if (!st.empty()) ans[i] = st.top();
        st.push(a[i]);
    }
    return ans;
}

In [ ]:
// Tests: input -> actual (Expected: X)
{
    auto r1 = nextSmaller({4,8,5,2,25});
    printVec("nextSmaller [4,8,5,2,25] ->", r1); cout << " (Expected: [2,5,2,-1,-1])\n";
    auto r2 = nextSmaller({1,2,3,4});
    printVec("nextSmaller [1,2,3,4] ->", r2); cout << " (Expected: [-1,-1,-1,-1])\n";
    auto r3 = nextSmaller({4,3,2,1});
    printVec("nextSmaller [4,3,2,1] ->", r3); cout << " (Expected: [3,2,1,-1])\n";
}

## 4. Count of Greater Elements to the Right

For each $i$, count how many $j > i$ have $a[j] > a[i]$. Unlike "next greater" (a *single* element), this is a *count*, so a plain monotonic stack does not suffice — it discards popped elements. We need an order-statistics structure.

### Approach: Binary Indexed Tree (Fenwick) over rank, scanning R→L
Coordinate-compress values to ranks $1..m$. Scan right to left; for each $a[i]$, query the BIT for the count of already-seen elements with rank **strictly greater** than rank$(a[i])$, then insert $a[i]$.

$$\text{count}[i] = \sum_{r = \text{rank}(a[i])+1}^{m} \text{seen}[r], \qquad O(n \log n).$$

(This problem is included because the *naming* invites a monotonic stack, but the count requirement is the delta: monotonic stacks answer "the next/nearest extreme," never "how many extremes." Recognizing that boundary is the lesson.)

In [ ]:
struct BIT {
    vector<int> t; int n;
    BIT(int n): t(n+1, 0), n(n) {}
    void add(int i, int v) { for (; i <= n; i += i & (-i)) t[i] += v; }
    int  sum(int i) const { int s=0; for (; i>0; i -= i & (-i)) s += t[i]; return s; } // [1..i]
};

vector<int> countGreaterToRight(const vector<int>& a) {
    int n = a.size();
    vector<int> sorted(a);
    sort(sorted.begin(), sorted.end());
    sorted.erase(unique(sorted.begin(), sorted.end()), sorted.end());
    int m = sorted.size();
    auto rank = [&](int x){ return int(lower_bound(sorted.begin(), sorted.end(), x) - sorted.begin()) + 1; };

    BIT bit(m);
    vector<int> ans(n, 0);
    for (int i = n - 1; i >= 0; --i) {
        int r = rank(a[i]);
        ans[i] = bit.sum(m) - bit.sum(r);   // count of seen ranks strictly > r
        bit.add(r, 1);                       // record a[i] as seen
    }
    return ans;
}

In [ ]:
// Tests: input -> actual (Expected: X)
{
    auto r1 = countGreaterToRight({3,1,2,4});
    // 3: {4} ->1 ; 1: {2,4} ->2 ; 2: {4} ->1 ; 4: {} ->0
    printVec("countGreaterRight [3,1,2,4] ->", r1); cout << " (Expected: [1,2,1,0])\n";
    auto r2 = countGreaterToRight({5,4,3,2,1});
    printVec("countGreaterRight [5,4,3,2,1] ->", r2); cout << " (Expected: [0,0,0,0,0])\n";
    auto r3 = countGreaterToRight({1,2,3,4,5});
    printVec("countGreaterRight [1,2,3,4,5] ->", r3); cout << " (Expected: [4,3,2,1,0])\n";
}

## 5. Trapping Rain Water — three approaches

Water above column $i$ is bounded by the tallest wall on each side:
$$\text{water}[i] = \max\!\big(0,\ \min(\text{leftMax}[i], \text{rightMax}[i]) - h[i]\big).$$

### Approach A — prefix/suffix max arrays, $O(n)$ time, $O(n)$ space
Precompute `leftMax` and `rightMax`, then sum the formula. Direct transcription of the definition.

### Approach B — two pointers, $O(n)$ time, $O(1)$ space
**Invariant:** maintain `l`, `r`, `leftMax`, `rightMax`. The key fact: if `leftMax <= rightMax`, then for column `l` the water is determined by `leftMax` *alone* — because some wall on the right is already $\ge$ `leftMax`, so $\min(\text{leftMax}, \text{rightMax}) = \text{leftMax}$ at `l` regardless of the exact `rightMax[l]`. Move the smaller side inward.

$$\text{if } \text{leftMax} \le \text{rightMax}:\ \text{water}[l] = \text{leftMax} - h[l],\ l{+}{+}; \quad \text{else symmetric on } r.$$

**Proof of the invariant:** the side with the smaller running max is *safe* to finalize, since the opposite running max is a lower bound on the true opposite max, and it already dominates — so the true min equals the smaller running max. $\square$

### Approach C — monotonic (decreasing) stack, $O(n)$ time, $O(n)$ space
Maintain a decreasing stack of indices. When `h[i]` exceeds the top, the popped bar is a *basin floor*; water fills between the new top (left wall) and `i` (right wall), bounded by $\min$ of the two walls minus the floor, times the width. This sums water in **horizontal layers**.

In [ ]:
// A: prefix/suffix arrays
long long trapPrefix(const vector<int>& h) {
    int n = h.size(); if (n == 0) return 0;
    vector<int> L(n), R(n);
    L[0] = h[0]; for (int i=1;i<n;++i) L[i] = max(L[i-1], h[i]);
    R[n-1] = h[n-1]; for (int i=n-2;i>=0;--i) R[i] = max(R[i+1], h[i]);
    long long w = 0;
    for (int i=0;i<n;++i) w += min(L[i], R[i]) - h[i];   // >=0 since L[i],R[i]>=h[i]
    return w;
}

// B: two pointers, O(1) space
long long trapTwoPointer(const vector<int>& h) {
    int n = h.size(); if (n == 0) return 0;
    int l = 0, r = n - 1, leftMax = 0, rightMax = 0;
    long long w = 0;
    while (l < r) {
        if (h[l] <= h[r]) {                 // left side has the binding (smaller) wall
            leftMax = max(leftMax, h[l]);
            w += leftMax - h[l];            // safe: rightMax >= h[r] >= h[l] side dominates
            ++l;
        } else {
            rightMax = max(rightMax, h[r]);
            w += rightMax - h[r];
            --r;
        }
    }
    return w;
}

// C: monotonic decreasing stack, layer by layer
long long trapStack(const vector<int>& h) {
    int n = h.size();
    stack<int> st;                          // indices, heights decreasing
    long long w = 0;
    for (int i = 0; i < n; ++i) {
        while (!st.empty() && h[i] > h[st.top()]) {
            int floor = st.top(); st.pop(); // this bar is the basin floor
            if (st.empty()) break;          // no left wall -> water spills out
            int left = st.top();
            int width = i - left - 1;       // columns strictly between the two walls
            int bounded = min(h[left], h[i]) - h[floor];  // layer height above floor
            w += (long long)width * bounded;
        }
        st.push(i);
    }
    return w;
}

In [ ]:
// Tests: input -> actual (Expected: X) -- all three must agree
{
    vector<int> h1 = {0,1,0,2,1,0,1,3,2,1,2,1};   // classic, answer 6
    vector<int> h2 = {4,2,0,3,2,5};               // answer 9
    vector<int> h3 = {3,0,2,0,4};                 // answer 7
    for (auto& [name, h, exp] : vector<tuple<string,vector<int>,long long>>{
            {"classic", h1, 6}, {"4,2,0,3,2,5", h2, 9}, {"3,0,2,0,4", h3, 7}}) {
        cout << name << ": prefix=" << trapPrefix(h)
             << " twoPtr=" << trapTwoPointer(h)
             << " stack=" << trapStack(h)
             << " (Expected: " << exp << ")\n";
    }
    cout << "empty -> " << trapTwoPointer({}) << " (Expected: 0)\n";
    cout << "single -> " << trapTwoPointer({5}) << " (Expected: 0)\n";
    cout << "monotone -> " << trapTwoPointer({1,2,3,4}) << " (Expected: 0)\n";
}

## 6. Sum of Subarray Minimums — the Contribution Technique

Sum, over **all** $O(n^2)$ subarrays, of the minimum. Brute force is $O(n^2)$ (or worse). The trick: don't iterate subarrays — for each element $a[i]$, count exactly **how many subarrays have $a[i]$ as their minimum**, and weight by $a[i]$.

### Contribution formula
Let $L[i]$ = number of consecutive elements ending at $i$ (going left) that are $> a[i]$ — i.e. distance to the **previous strictly smaller** element. Let $R[i]$ = distance to the **next smaller-or-equal** element (going right). Then $a[i]$ is the minimum of exactly $L[i] \cdot R[i]$ subarrays:

$$\text{answer} = \sum_{i} a[i] \cdot L[i] \cdot R[i].$$

### The strict/non-strict asymmetry (the critical delta)
When equal values appear, a subarray's minimum is achieved by several elements — we must attribute each subarray to **exactly one** of them or we double-count. The fix: make one boundary **strict** and the other **non-strict**. Convention used here:
- left boundary: **previous strictly smaller** (`< a[i]`) → the left while-loop pops on `>=`,
- right boundary: **next smaller-or-equal** (`<= a[i]`) → the right while-loop pops on `>`.

This assigns each tie-block's subarrays to the **leftmost** minimal element only, eliminating double-counting. $\square$

### Boundary transitions
$L[i] = i - \text{PSE}(i)$, $R[i] = \text{NSEorEq}(i) - i$, with sentinels $\text{PSE}=-1$ and $\text{NSEorEq}=n$ for "none".

In [ ]:
const long long MOD = 1000000007LL;

long long sumSubarrayMins(const vector<int>& a) {
    int n = a.size();
    vector<int> L(n), R(n);
    // L[i] = i - (index of previous STRICTLY smaller); pop on >= (treat equals as not-smaller)
    {
        stack<int> st;
        for (int i = 0; i < n; ++i) {
            while (!st.empty() && a[st.top()] >= a[i]) st.pop();
            L[i] = st.empty() ? (i + 1) : (i - st.top());
            st.push(i);
        }
    }
    // R[i] = (index of next SMALLER-OR-EQUAL) - i; pop on > (equals belong to the right span)
    {
        stack<int> st;
        for (int i = n - 1; i >= 0; --i) {
            while (!st.empty() && a[st.top()] > a[i]) st.pop();
            R[i] = st.empty() ? (n - i) : (st.top() - i);
            st.push(i);
        }
    }
    long long ans = 0;
    for (int i = 0; i < n; ++i)
        ans = (ans + (long long)a[i] % MOD * L[i] % MOD * R[i]) % MOD;
    return ans;
}

In [ ]:
// Tests: input -> actual (Expected: X)
{
    cout << "[3,1,2,4] -> " << sumSubarrayMins({3,1,2,4}) << " (Expected: 17)\n";
    cout << "[11,81,94,43,3] -> " << sumSubarrayMins({11,81,94,43,3}) << " (Expected: 444)\n";
    cout << "[3,3,3] -> " << sumSubarrayMins({3,3,3}) << " (Expected: 18)\n"; // ties handled
    cout << "[1] -> " << sumSubarrayMins({1}) << " (Expected: 1)\n";
    cout << "[2,9,7,8,3,4,6,1] -> " << sumSubarrayMins({2,9,7,8,3,4,6,1}) << " (Expected: 117)\n";
}

## 7. Sum of Subarray Ranges

Range of a subarray = max − min. Summed over all subarrays:
$$\sum_{\text{subarrays}} (\max - \min) = \underbrace{\sum_{\text{subarrays}} \max}_{\text{contribution by max}} - \underbrace{\sum_{\text{subarrays}} \min}_{\text{contribution by min}}.$$

So it's **two** contribution computations: sum-of-maxima minus sum-of-minima, each via the identical monotonic-stack span technique (flip comparisons for the max version). The strict/non-strict tie convention must be applied consistently to both so equal values aren't double-counted.

$$\text{answer} = \sum_i a[i]\,L^{\max}_i R^{\max}_i - \sum_i a[i]\,L^{\min}_i R^{\min}_i.$$

In [ ]:
long long sumSubarrayRanges(const vector<int>& a) {
    int n = a.size();
    auto spanSum = [&](bool wantMax) -> long long {
        vector<int> L(n), R(n);
        // For MAX: previous strictly greater (left), next greater-or-equal (right)
        // For MIN: previous strictly smaller (left), next smaller-or-equal (right)
        {
            stack<int> st;
            for (int i = 0; i < n; ++i) {
                while (!st.empty() &&
                       (wantMax ? a[st.top()] <= a[i] : a[st.top()] >= a[i])) st.pop();
                L[i] = st.empty() ? (i + 1) : (i - st.top());
                st.push(i);
            }
        }
        {
            stack<int> st;
            for (int i = n - 1; i >= 0; --i) {
                while (!st.empty() &&
                       (wantMax ? a[st.top()] < a[i] : a[st.top()] > a[i])) st.pop();
                R[i] = st.empty() ? (n - i) : (st.top() - i);
                st.push(i);
            }
        }
        long long s = 0;
        for (int i = 0; i < n; ++i) s += (long long)a[i] * L[i] * R[i];
        return s;
    };
    return spanSum(true) - spanSum(false);   // sum of maxima - sum of minima
}

In [ ]:
// Tests: input -> actual (Expected: X)
{
    cout << "[1,2,3] -> " << sumSubarrayRanges({1,2,3}) << " (Expected: 4)\n";
    cout << "[1,3,3] -> " << sumSubarrayRanges({1,3,3}) << " (Expected: 4)\n"; // ties
    cout << "[4,-2,-3,4,1] -> " << sumSubarrayRanges({4,-2,-3,4,1}) << " (Expected: 59)\n";
    cout << "[5] -> " << sumSubarrayRanges({5}) << " (Expected: 0)\n";
}

## 8. Asteroid Collision

Positive = moving right, negative = moving left; magnitude = size. A right-mover followed by a left-mover collide; the smaller explodes, equal sizes both explode.

### Invariant
The stack holds the **surviving** asteroids in order. A collision is possible **only** when the incoming asteroid is left-moving (`<0`) and the stack top is right-moving (`>0`) — the only configuration where two asteroids approach each other. All other adjacencies are stable.

### The four cases when `top > 0` and `incoming < 0`
| Comparison (`top` vs `|incoming|`) | Action |
|---|---|
| `top < |incoming|` | top explodes (pop); incoming continues — keep colliding with new top |
| `top == |incoming|` | both explode (pop top); incoming destroyed — stop |
| `top > |incoming|` | incoming explodes — stop, push nothing |
| stack empty / top < 0 (left-mover) | no collision; push incoming |

The `while` loop processes a chain of collisions for one incoming asteroid; the amortized bound holds because each asteroid is pushed/popped at most once.

In [ ]:
vector<int> asteroidCollision(const vector<int>& asteroids) {
    vector<int> st;                          // survivors, as a stack
    for (int x : asteroids) {
        bool alive = true;
        // collision only if x moves left and top moves right
        while (alive && x < 0 && !st.empty() && st.back() > 0) {
            if (st.back() < -x)      { st.pop_back(); }      // top smaller -> explodes, x continues
            else if (st.back() == -x){ st.pop_back(); alive = false; } // equal -> both explode
            else                     { alive = false; }      // top bigger -> x explodes
        }
        if (alive) st.push_back(x);          // survived all collisions (or never collided)
    }
    return st;
}

In [ ]:
// Tests: input -> actual (Expected: X)
{
    auto r1 = asteroidCollision({5,10,-5});
    printVec("[5,10,-5] ->", r1); cout << " (Expected: [5,10])\n";
    auto r2 = asteroidCollision({8,-8});
    printVec("[8,-8] ->", r2); cout << " (Expected: [])\n";
    auto r3 = asteroidCollision({10,2,-5});
    printVec("[10,2,-5] ->", r3); cout << " (Expected: [10])\n";
    auto r4 = asteroidCollision({-2,-1,1,2});
    printVec("[-2,-1,1,2] ->", r4); cout << " (Expected: [-2,-1,1,2])\n"; // none collide
    auto r5 = asteroidCollision({-2,2,-1,-2});
    printVec("[-2,2,-1,-2] ->", r5); cout << " (Expected: [-2])\n"; // 2 kills -1; then -2==2 both die
}

## 9. Remove K Digits

Given a numeric string, remove exactly $k$ digits to make the smallest possible number.

### Greedy via increasing monotonic stack
**Key lemma:** to minimize, whenever a digit is **larger** than the next digit, removing the larger one (the earlier, more-significant position) yields a strictly smaller number. So we scan left to right and pop any stack top that is greater than the incoming digit (while budget $k$ remains) — building a non-decreasing stack.

*Proof of lemma:* removing a digit at a more significant position has greater effect on magnitude. If `s[j] > s[j+1]`, deleting `s[j]` makes the digit at that position smaller (it becomes `s[j+1]`), strictly reducing the number; deleting any later digit cannot reduce that more-significant position. So the greedy removal is optimal at each step. $\square$

### Two-phase cleanup
1. **Pop phase:** while scanning, pop greater tops until $k$ exhausted.
2. **Trailing phase:** if $k$ remains after the scan (string was non-decreasing), remove the last $k$ digits (the largest, least-significant).
3. **Leading-zero phase:** strip leading zeros from the result; empty result becomes `"0"`.

### Boundary transitions
| Incoming digit `d` | Action |
|---|---|
| stack top `> d` and `k>0` | pop, `k--` |
| otherwise | push `d` |
| after scan, `k>0` | drop last `k` |
| build result | strip leading zeros, default `"0"` |

In [ ]:
string removeKdigits(const string& num, int k) {
    string st;                               // increasing-ish stack of kept digits
    for (char d : num) {
        while (!st.empty() && k > 0 && st.back() > d) {  // larger, more-significant -> drop
            st.pop_back(); --k;
        }
        st.push_back(d);
    }
    while (k > 0 && !st.empty()) { st.pop_back(); --k; } // leftover removals from the end
    int i = 0; while (i < (int)st.size() && st[i] == '0') ++i;  // strip leading zeros
    string res = st.substr(i);
    return res.empty() ? "0" : res;
}

In [ ]:
// Tests: input -> actual (Expected: X)
{
    cout << "1432219, k=3 -> " << removeKdigits("1432219", 3) << " (Expected: 1219)\n";
    cout << "10200,   k=1 -> " << removeKdigits("10200", 1)   << " (Expected: 200)\n";
    cout << "10,      k=2 -> " << removeKdigits("10", 2)      << " (Expected: 0)\n";  // remove all
    cout << "112,     k=1 -> " << removeKdigits("112", 1)     << " (Expected: 11)\n"; // trailing phase
    cout << "1234567, k=3 -> " << removeKdigits("1234567", 3) << " (Expected: 1234)\n"; // non-decr
    cout << "9,       k=1 -> " << removeKdigits("9", 1)       << " (Expected: 0)\n";
}

## 10. Largest Rectangle in a Histogram

For each bar $i$, the largest rectangle *with $i$ as the limiting (shortest) height* extends left to the **previous smaller** bar and right to the **next smaller** bar. Its area:
$$\text{area}_i = h[i] \cdot (R[i] - L[i] - 1),$$
where $L[i]$ = index of previous strictly smaller (or $-1$), $R[i]$ = index of next strictly smaller (or $n$). The answer is $\max_i \text{area}_i$.

### Why the sentinels $-1$ and $n$
A bar with no smaller bar to its left has the whole left side available → left boundary is $-1$, so width counts from index $0$. Symmetrically, no smaller bar to the right → right boundary $n$. These sentinels make the width formula uniform with no special cases.

### Single-pass stack approach (the elegant one)
Maintain an **increasing** stack of indices. When bar $i$ is shorter than the top, the popped bar's **next smaller** is $i$ and its **previous smaller** is the new top — both boundaries known at the moment of the pop, so we compute its area immediately:
$$\text{width} = i - \text{st.top()} - 1 \quad(\text{or } i \text{ if stack empties}),\qquad \text{area} = h[\text{popped}]\cdot \text{width}.$$
Appending a sentinel height $0$ at the end flushes the stack cleanly.

In [ ]:
long long largestRectangleArea(vector<int> h) {
    h.push_back(0);                          // sentinel forces a final flush of the stack
    int n = h.size();
    stack<int> st;                           // indices, heights increasing bottom->top
    long long best = 0;
    for (int i = 0; i < n; ++i) {
        while (!st.empty() && h[st.top()] >= h[i]) {  // i is the NEXT SMALLER of top
            int height = h[st.top()]; st.pop();
            int left = st.empty() ? -1 : st.top();    // PREVIOUS SMALLER = new top (or -1)
            long long width = i - left - 1;           // bars strictly between boundaries
            best = max(best, (long long)height * width);
        }
        st.push(i);
    }
    return best;
}

In [ ]:
// Tests: input -> actual (Expected: X)
{
    cout << "[2,1,5,6,2,3] -> " << largestRectangleArea({2,1,5,6,2,3}) << " (Expected: 10)\n";
    cout << "[2,4]         -> " << largestRectangleArea({2,4})         << " (Expected: 4)\n";
    cout << "[6,2,5,4,5,1,6] -> " << largestRectangleArea({6,2,5,4,5,1,6}) << " (Expected: 12)\n";
    cout << "[5,5,5,5]     -> " << largestRectangleArea({5,5,5,5})     << " (Expected: 20)\n"; // all-same
    cout << "[1]           -> " << largestRectangleArea({1})           << " (Expected: 1)\n";
    cout << "[2,1,2]       -> " << largestRectangleArea({2,1,2})       << " (Expected: 3)\n";
}

## 11. Maximal Rectangle

Given a binary matrix, find the largest all-`1` rectangle.

### Reduction to histogram, row by row
Process rows top to bottom, maintaining a `heights` array: `heights[c]` = number of consecutive `1`s ending at the current row in column `c`.
$$\text{heights}[c] = \begin{cases} \text{heights}[c] + 1 & \text{cell} = 1 \\ 0 & \text{cell} = 0 \end{cases}$$
After updating each row, the largest all-`1` rectangle whose **bottom edge lies on that row** equals the largest rectangle in the histogram `heights`. Taking the max over all rows gives the global answer.

### Why the reduction is correct
Any all-`1` rectangle has a bottom row $r$; at row $r$ its columns all have `heights[c] >= (rectangle height)`, so it appears as a rectangle in row $r$'s histogram. Conversely every histogram rectangle corresponds to an all-`1` submatrix (the consecutive-`1`s definition guarantees the cells above are all `1`). So scanning every row's histogram covers exactly all candidate rectangles. $\square$

$$\text{Complexity: } O(mn) \text{ — each of } m \text{ rows runs an } O(n) \text{ histogram pass.}$$

In [ ]:
long long maximalRectangle(const vector<vector<char>>& matrix) {
    if (matrix.empty() || matrix[0].empty()) return 0;
    int rows = matrix.size(), cols = matrix[0].size();
    vector<int> heights(cols, 0);
    long long best = 0;
    for (int r = 0; r < rows; ++r) {
        for (int c = 0; c < cols; ++c)
            heights[c] = (matrix[r][c] == '1') ? heights[c] + 1 : 0;  // accumulate consecutive 1s
        best = max(best, largestRectangleArea(heights));              // bottom edge on row r
    }
    return best;
}

In [ ]:
// Tests: input -> actual (Expected: X)
{
    vector<vector<char>> m1 = {
        {'1','0','1','0','0'},
        {'1','0','1','1','1'},
        {'1','1','1','1','1'},
        {'1','0','0','1','0'}};
    cout << "matrix1 -> " << maximalRectangle(m1) << " (Expected: 6)\n";

    vector<vector<char>> m2 = {{'0'}};
    cout << "[[0]] -> " << maximalRectangle(m2) << " (Expected: 0)\n";

    vector<vector<char>> m3 = {{'1'}};
    cout << "[[1]] -> " << maximalRectangle(m3) << " (Expected: 1)\n";

    vector<vector<char>> m4 = {{'1','1'},{'1','1'}};
    cout << "all-ones 2x2 -> " << maximalRectangle(m4) << " (Expected: 4)\n";
}

## Unified Mental Model

```text
                  ONE STACK, ELEVEN QUESTIONS
                  ===========================
  Store INDICES. Maintain monotone order. A POP is an ANSWER.

  order=DECREASING, scan L->R : pop -> popped's NEXT GREATER(-eq) is i
  order=INCREASING, scan L->R : pop -> popped's NEXT SMALLER(-eq) is i
  after a pop, the NEW TOP    : popped's PREVIOUS extreme (left boundary)

  ┌─ single nearest extreme ─────────► NGE / NSE / NGE-circular (2n, i%n)
  ├─ COUNT of extremes ──────────────► NOT a mono stack → BIT/Fenwick
  ├─ water between walls ────────────► trap: prefix | two-ptr O(1) | stack layers
  ├─ each elem's DOMINION (span) ────► contribution: a[i]*L[i]*R[i]
  │     SubarrayMins  = sum of (min contributions)
  │     SubarrayRanges= sum(max contrib) - sum(min contrib)
  │     strict on ONE side, non-strict on OTHER  => no double count
  ├─ rectangle under bars ───────────► histogram: area = h*(R-L-1), sentinels -1,n
  │     MaximalRectangle = histogram per row (heights accumulate 1s)
  ├─ lexicographic minimization ─────► RemoveKDigits: pop bigger-then-smaller, greedy
  └─ approaching objects collide ────► Asteroids: collide only (top>0, incoming<0)
```

**The single unifying move:** restoring monotonic order is not overhead — the *displacement* during restoration is the computation. Whatever the question (nearest extreme, span, water layer, rectangle boundary, greedy removal), the moment an element leaves the stack, the geometry around it is fully determined.

## Decision Tree — Which Variant?

```text
What does each element need to know about its neighbors?
│
├─ "nearest STRICTLY greater/smaller, one element"
│     ├─ to the right, linear        → NGE / NSE  (R->L scan, store values)
│     └─ circular array              → NGE II     (loop 2n, index % n, write if i<n)
│
├─ "HOW MANY are greater to my right" (a count, not one)
│     → monotonic stack INSUFFICIENT → Fenwick/BIT over compressed ranks, R->L
│
├─ "how much water sits above me"
│     ├─ O(1) space required         → two pointers (move smaller wall in)
│     ├─ clarity over space          → prefix/suffix max arrays
│     └─ layer-by-layer reasoning    → decreasing stack
│
├─ "in how many subarrays am I the min/max" (aggregate)
│     → contribution technique: L[i]*R[i]; strict one side, non-strict other
│       ├─ sum of minimums           → min spans
│       └─ sum of ranges             → max spans - min spans
│
├─ "largest rectangle bounded by smaller neighbors"
│     ├─ 1-D histogram               → increasing stack, area h*(R-L-1)
│     └─ 2-D binary matrix           → per-row histogram reduction (O(mn))
│
├─ "smallest number after k removals"
│     → greedy increasing stack, pop bigger predecessors, 2-phase + zero strip
│
└─ "objects moving toward each other survive?"
      → stack of survivors; collide iff (top right-moving, incoming left-moving)
```

## Complexity Summary

| Problem | Time | Space | Order / Tool | Key Invariant |
|---|---|---|---|---|
| Next Greater Element (I) | $O(n)$ | $O(n)$ | decreasing, R→L | stack = decreasing candidates; top = next greater |
| NGE II (circular) | $O(n)$ | $O(n)$ | decreasing, $2n$, `i%n` | second-lap seeds wrap candidates; write only `i<n` |
| Next Smaller Element | $O(n)$ | $O(n)$ | increasing, R→L | stack = increasing candidates; top = next smaller |
| Count Greater to Right | $O(n\log n)$ | $O(n)$ | Fenwick/BIT, R→L | order-statistics, not a mono stack |
| Trapping Rain Water | $O(n)$ | $O(1)$–$O(n)$ | two-ptr / prefix / stack | water = $\min(\text{Lmax},\text{Rmax})-h$ |
| Sum of Subarray Minimums | $O(n)$ | $O(n)$ | contribution | $\sum a[i]L[i]R[i]$; strict left / non-strict right |
| Sum of Subarray Ranges | $O(n)$ | $O(n)$ | contribution ×2 | sum(max spans) − sum(min spans) |
| Asteroid Collision | $O(n)$ | $O(n)$ | survivor stack | collide iff top>0 and incoming<0 |
| Remove K Digits | $O(n)$ | $O(n)$ | increasing greedy | pop larger more-significant digit first |
| Largest Rectangle (Histogram) | $O(n)$ | $O(n)$ | increasing | area $=h(R-L-1)$, sentinels $-1,n$ |
| Maximal Rectangle | $O(mn)$ | $O(n)$ | per-row histogram | heights accumulate consecutive 1s |